# Weather Forecasting 

predict next-hour temperature using past weather data.
All data work runs in Snowpark — no pandas for processing.

**Steps**
1. Load data
2. Data cleaning (5 simple steps)
3. Feature engineering (using multiple sensors, not just temperature) + plots
4. Train/test split (by time, not random)
5. Train one model
6. save test data as CSV



## Step 0 — Connect

Uses the same `get_session()` helper you already have.

In [2]:
# ==========================================================
# Project Imports
# ==========================================================

import sys
from pathlib import Path
# --- Path setup (so Python can find your connection helper) -------------
import sys
from pathlib import Path

# --- Snowpark core --------------------------------------------------------
from snowflake.snowpark import Window
from snowflake.snowpark import functions as F

# --- Snowpark ML: model + evaluation metrics ------------------------------
from snowflake.ml.modeling.ensemble import RandomForestRegressor


# --- Plotting (Plotly) -----------------------------------------------------
import plotly.express as px
import plotly.graph_objects as go

project_root = Path.cwd().parent

if str(project_root / "Snowpark") not in sys.path:
    sys.path.append(str(project_root / "Snowpark"))

In [3]:
# ==========================================================
# Create Snowflake Session
# File: src/utils/snowflake_connection.py
# ==========================================================
# --- Snowflake connection helper -----------------------------------------
from src.utils.snowflake_connection import get_session

session = get_session("dev")

session

try init connection snowflake


## Step 1 — Load data

`session.table()` just points at the table — no data is pulled yet. `.show()` runs a small
query so you can see what you're working with.


In [5]:
SOURCE_TABLE = "TBL_WEATHER_DATA"
TARGET = "TEMPERATURE"
def load_data(session, table_name):
    """Load the raw weather table and normalize column names to uppercase."""
    df = session.table(table_name)
    df = df.select([F.col(c).alias(c.upper()) for c in df.columns])
    return df

df = load_data(session, SOURCE_TABLE)
df.show(5)

---------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TIMESTAMP"       |"TEMPERATURE"  |"HUMIDITY"  |"PRESSURE"  |"WIND_SPEED"  |"PRECIPITATION"  |"CLOUD_COVER"  |"LATITUDE"  |"LONGITUDE"  |"TIMEZONE"  |"ELEVATION"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-07-17T00:00  |24.7           |78          |1007.4      |11.0          |0.1              |100            |17.398945   |78.457085    |GMT         |505.0        |
|2026-07-17T01:00  |24.5           |80          |1008.1      |10.4          |0.1              |100            |17.398945   |78.457085    |GMT         |505.0        |
|2026-07-17T02:00  |25.0           |79          |1008.6      |11.6          |0.2              |100            |17.398945   |78.457085    |GMT         |505.0        |
|202

## Step 2 — Data cleaning (5 simple steps)

### 2.1 Fix data types

`timestamp` often comes in as text. We convert it to a real timestamp type so it can be
sorted and used in time calculations (lags, hour extraction, etc.).

In [6]:
def fix_data_types(df):
    """Convert the timestamp column from text to a real timestamp type."""
    return df.with_column("TIMESTAMP", F.to_timestamp_ntz(F.col("TIMESTAMP")))

df_step1 = fix_data_types(df)
df_step1.select("TIMESTAMP").show(3)

-----------------------
|"TIMESTAMP"          |
-----------------------
|2026-07-17 00:00:00  |
|2026-07-17 01:00:00  |
|2026-07-17 02:00:00  |
-----------------------



### 2.2 Check and handle missing values

We count nulls in every column. Any row with a missing `TEMPERATURE` (our target) has to be
dropped — we can't train on an unknown answer.

In [7]:
def check_missing_values(df):
    """Print how many nulls each column has."""
    null_counts = df.agg(
        *[F.sum(F.iff(F.col(c).is_null(), 1, 0)).alias(c) for c in df.columns]
    ).collect()[0].as_dict()

    print("Missing values per column:")
    for col, count in null_counts.items():
        print(f"  {col}: {count or 0}")
    return null_counts

def drop_missing_target(df, target_col):
    """Drop rows where the target value is missing."""
    return df.filter(F.col(target_col).is_not_null())

_ = check_missing_values(df_step1)
df_step2 = drop_missing_target(df_step1, TARGET)
print("\nRows after dropping missing targets:", df_step2.count())

Missing values per column:
  TEMPERATURE: 0
  HUMIDITY: 0
  PRESSURE: 0
  WIND_SPEED: 0
  PRECIPITATION: 0
  CLOUD_COVER: 0
  LATITUDE: 0
  LONGITUDE: 0
  TIMEZONE: 0
  ELEVATION: 0
  TIMESTAMP: 0

Rows after dropping missing targets: 1464


### 2.3 Check and remove duplicate rows

Duplicate rows can quietly bias a model (the same example gets counted twice) and can mess
up a time-based train/test split.

In [8]:
def check_and_remove_duplicates(df):
    """Print how many duplicate rows exist, then remove them."""
    total = df.count()
    unique = df.distinct().count()
    print(f"Total rows: {total} | Unique rows: {unique} | Duplicates removed: {total - unique}")
    return df.distinct()

df_step3 = check_and_remove_duplicates(df_step2)

Total rows: 1464 | Unique rows: 1464 | Duplicates removed: 0


### 2.4 Drop irrelevant columns

`LATITUDE`, `LONGITUDE`, `TIMEZONE`, and `ELEVATION` describe the weather **station**, not
the weather itself. Since this data comes from a single fixed location, these columns never
change — a column that never changes can't help predict anything, so it's dropped.

In [9]:
def drop_irrelevant_columns(df, columns_to_drop):
    """Drop columns that carry no predictive value (e.g. constant station metadata)."""
    existing = [c for c in columns_to_drop if c in df.columns]
    print("Dropping columns:", existing)
    return df.drop(*existing) if existing else df

COLUMNS_TO_DROP = ["LATITUDE", "LONGITUDE", "TIMEZONE", "ELEVATION"]
df_step4 = drop_irrelevant_columns(df_step3, COLUMNS_TO_DROP)
print("\nRemaining columns:", df_step4.columns)

Dropping columns: ['LATITUDE', 'LONGITUDE', 'TIMEZONE', 'ELEVATION']

Remaining columns: ['TEMPERATURE', 'HUMIDITY', 'PRESSURE', 'WIND_SPEED', 'PRECIPITATION', 'CLOUD_COVER', 'TIMESTAMP']


### 2.5 Sort by time

Every later step — lag features, the train/test split — assumes the rows are in
chronological order.

In [10]:
def sort_by_time(df):
    """Sort rows chronologically. Required before creating lag features."""
    return df.sort(F.col("TIMESTAMP").asc())

df_clean = sort_by_time(df_step4)
df_clean.show(5)

------------------------------------------------------------------------------------------------------------------
|"TEMPERATURE"  |"HUMIDITY"  |"PRESSURE"  |"WIND_SPEED"  |"PRECIPITATION"  |"CLOUD_COVER"  |"TIMESTAMP"          |
------------------------------------------------------------------------------------------------------------------
|24.7           |78          |1007.4      |11.0          |0.1              |100            |2026-07-17 00:00:00  |
|24.5           |80          |1008.1      |10.4          |0.1              |100            |2026-07-17 01:00:00  |
|25.0           |79          |1008.6      |11.6          |0.2              |100            |2026-07-17 02:00:00  |
|25.4           |79          |1009.0      |13.1          |0.2              |100            |2026-07-17 03:00:00  |
|25.4           |81          |1009.3      |11.4          |0.4              |100            |2026-07-17 04:00:00  |
--------------------------------------------------------------------------------

## Step 3 — Feature engineering

**The key idea for forecasting:** at prediction time, you only know the *past*. So every
feature must use data from **before** the hour we're predicting — never the same hour,
never the future.

Using only temperature's own lags would work, but leaves useful signal on the table. We also
bring in the **past** values of the other sensors — humidity, pressure, wind speed,
precipitation, and cloud cover — since each of these physically relates to how temperature
moves. The important word is *past*: we use last hour's humidity, not this hour's, since
this hour's humidity wouldn't actually be known yet in a real forecast.

In [11]:
def add_features(df, target_col, other_cols, n_lags=3):
    """
    Add time-series features:
      - lag_1, lag_2, lag_3 of the target (temperature)
      - lag_1 of each other sensor column (humidity, pressure, etc.)
      - hour of day (0-23), which captures the daily warm/cool cycle
    All features use only past values -- never the current hour's target or sensors.
    """
    time_window = Window.order_by(F.col("TIMESTAMP").asc())

    # Lag features for the target
    for lag in range(1, n_lags + 1):
        df = df.with_column(f"{target_col}_LAG_{lag}",
                            F.lag(F.col(target_col), lag).over(time_window))

    # Lag-1 features for the other sensors (their most recent past reading)
    for col in other_cols:
        df = df.with_column(f"{col}_LAG_1", F.lag(F.col(col), 1).over(time_window))

    # Hour of day -- known in advance, so it's safe to use directly
    df = df.with_column("HOUR", F.hour(F.col("TIMESTAMP")))

    return df

OTHER_SENSOR_COLS = ["HUMIDITY", "PRESSURE", "WIND_SPEED", "PRECIPITATION", "CLOUD_COVER"]

df_features = add_features(df_clean, TARGET, OTHER_SENSOR_COLS, n_lags=3)
df_features.select("TIMESTAMP", TARGET, "TEMPERATURE_LAG_1", "HUMIDITY_LAG_1",
                   "PRESSURE_LAG_1", "HOUR").show(8)

------------------------------------------------------------------------------------------------------------
|"TIMESTAMP"          |"TEMPERATURE"  |"TEMPERATURE_LAG_1"  |"HUMIDITY_LAG_1"  |"PRESSURE_LAG_1"  |"HOUR"  |
------------------------------------------------------------------------------------------------------------
|2026-07-17 00:00:00  |24.7           |NULL                 |NULL              |NULL              |0       |
|2026-07-17 01:00:00  |24.5           |24.7                 |78                |1007.4            |1       |
|2026-07-17 02:00:00  |25.0           |24.5                 |80                |1008.1            |2       |
|2026-07-17 03:00:00  |25.4           |25.0                 |79                |1008.6            |3       |
|2026-07-17 04:00:00  |25.4           |25.4                 |79                |1009.0            |4       |
|2026-07-17 05:00:00  |25.9           |25.4                 |81                |1009.3            |5       |
|2026-07-17 06:00:0

Notice the first few rows show `null` for the lag columns — there's no "1 hour ago"
for the very first row in the data. Those rows can't be used for training, so we drop them.

In [12]:
def drop_warmup_rows(df):
    """Drop rows that don't have full lag history yet (nulls in any lag column)."""
    lag_cols = [c for c in df.columns if "_LAG_" in c]
    condition = F.lit(True)
    for c in lag_cols:
        condition = condition & F.col(c).is_not_null()
    return df.filter(condition)

df_ready = drop_warmup_rows(df_features)
print("Rows before:", df_features.count(), "| Rows after dropping warm-up rows:", df_ready.count())

Rows before: 1464 | Rows after dropping warm-up rows: 1461


### 3.1 A quick look at the data (Plotly)

A short but important note before plotting: **Plotly needs local data**, not a Snowpark
DataFrame. So for these plots — and only for these plots — we pull the (small) result down
with `.to_pandas()`. This is purely for visualization; the model itself trains directly on
the Snowpark DataFrame below, never on this pandas copy.

In [13]:
import plotly.express as px
import plotly.graph_objects as go

# Pull just the columns we need for plotting. Only for visualization -- not used in training.
plot_data = df_ready.select(
    "TIMESTAMP", TARGET, "TEMPERATURE_LAG_1", "HOUR", "HUMIDITY", "PRESSURE"
).to_pandas().sort_values("TIMESTAMP")

print("Rows pulled locally for plotting:", len(plot_data))
plot_data.head()

Rows pulled locally for plotting: 1461


,TIMESTAMP,TEMPERATURE,TEMPERATURE_LAG_1,HOUR,HUMIDITY,PRESSURE
0,2026-07-17 03:00:00,25.4,25.0,3,79,1009.0
1,2026-07-17 04:00:00,25.4,25.4,4,81,1009.3
2,2026-07-17 05:00:00,25.9,25.4,5,79,1009.4
3,2026-07-17 06:00:00,25.9,25.9,6,80,1009.3
4,2026-07-17 07:00:00,26.8,25.9,7,76,1008.7


**Plot 1 — Temperature over time.** The most basic check: does the data look like real
weather (smooth ups and downs), or are there odd jumps and gaps?

In [14]:
fig1 = px.line(plot_data, x="TIMESTAMP", y=TARGET, title="Temperature Over Time")
fig1.update_layout(xaxis_title="Time", yaxis_title="Temperature (°C)")
fig1.show()

**Plot 2 — Average temperature by hour of day.** Shows the daily cycle: usually cooler
in the early morning and warmer in the afternoon. This is *why* `HOUR` is a useful feature.

In [15]:
hourly_avg = plot_data.groupby("HOUR", as_index=False)[TARGET].mean()

fig2 = px.bar(hourly_avg, x="HOUR", y=TARGET, title="Average Temperature by Hour of Day")
fig2.update_layout(xaxis_title="Hour (0-23)", yaxis_title="Average Temperature (°C)")
fig2.show()

**Plot 3 — Temperature distribution.** A histogram shows the overall range and shape
of the values the model needs to predict.

In [16]:
fig3 = px.histogram(plot_data, x=TARGET, nbins=20, title="Distribution of Temperature Values")
fig3.update_layout(xaxis_title="Temperature (°C)", yaxis_title="Count")
fig3.show()

**Plot 4 — Temperature vs. 1-hour-ago temperature.** The strongest single feature. If
points roughly follow a diagonal line, last hour's temperature is a very good predictor.

In [17]:
fig4 = px.scatter(plot_data, x="TEMPERATURE_LAG_1", y=TARGET,
                  title="Temperature vs. Temperature 1 Hour Ago", opacity=0.6)
fig4.update_layout(xaxis_title="Temperature 1 Hour Ago (°C)", yaxis_title="Current Temperature (°C)")
fig4.show()

**Plot 5 — Humidity vs. temperature.** Since humidity is one of the extra features
we're now using, this shows the relationship: temperature tends to be higher when humidity
is lower.

In [18]:
fig5 = px.scatter(plot_data, x="HUMIDITY", y=TARGET,
                  title="Humidity vs. Temperature", opacity=0.6, color="HOUR")
fig5.update_layout(xaxis_title="Humidity (%)", yaxis_title="Temperature (°C)")
fig5.show()

## Step 4 — Choose forecast features (no data leakage)

This is a **forecasting** problem: predict a future temperature using only information
available beforehand. Same-hour sensor readings are **not allowed** as features — in real
forecasting you wouldn't have next hour's humidity reading yet either. That's why every
sensor column below is a `_LAG_1` (past) value, not the raw column.

In [19]:
FEATURE_COLS = (
    ["TEMPERATURE_LAG_1", "TEMPERATURE_LAG_2", "TEMPERATURE_LAG_3", "HOUR"]
    + [f"{c}_LAG_1" for c in OTHER_SENSOR_COLS]
)

print(f"Using {len(FEATURE_COLS)} features for forecasting:")
for c in FEATURE_COLS:
    print(" -", c)
print("\nTarget:", TARGET)

Using 9 features for forecasting:
 - TEMPERATURE_LAG_1
 - TEMPERATURE_LAG_2
 - TEMPERATURE_LAG_3
 - HOUR
 - HUMIDITY_LAG_1
 - PRESSURE_LAG_1
 - WIND_SPEED_LAG_1
 - PRECIPITATION_LAG_1
 - CLOUD_COVER_LAG_1

Target: TEMPERATURE


## Step 5 — Train / test split (by time)

For time-series data, we never shuffle. We train on the earlier part and test on the later
part — the same way a real forecast works: you only ever predict what comes *next*, using
what came *before*.

We use the first 80% of the timeline for training and the last 20% for testing.

In [20]:
from snowflake.snowpark import Window

def train_test_split_by_time(df, split_fraction=0.8):
    """Split chronologically using row position instead of PERCENTILE_CONT."""
    total_rows = df.count()
    split_point = int(total_rows * split_fraction)

    w = Window.order_by(F.col("TIMESTAMP").asc())
    df_numbered = df.with_column("ROW_NUM", F.row_number().over(w))

    train_df = df_numbered.filter(F.col("ROW_NUM") <= split_point).drop("ROW_NUM")
    test_df  = df_numbered.filter(F.col("ROW_NUM") >  split_point).drop("ROW_NUM")
    return train_df, test_df

In [21]:

train_df, test_df = train_test_split_by_time(df_ready)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 1168
Testing rows: 293


## Step 6 — Train one model

We use **Random Forest Regressor**. Reasons this is a good choice for a first model:

- Works well on tabular data with almost no tuning.
- Doesn't need feature scaling (unlike Linear Regression or KNN), which keeps the pipeline
  short even with more features.
- Handles the non-linear, seasonal pattern of temperature (via `HOUR`) and the mix of
  different sensor scales (humidity in %, pressure in hPa, etc.) without extra effort.



In [22]:
from snowflake.ml.modeling.ensemble import RandomForestRegressor

def train_model(train_df, feature_cols, target_col):
    """Train a Random Forest model to predict the target from the given features."""
    model = RandomForestRegressor(
        input_cols=feature_cols,
        label_cols=[target_col],
        output_cols=["PREDICTED_TEMPERATURE"],
        n_estimators=100,   # number of trees — 100 is a solid, simple default
        random_state=42,    # makes results reproducible
    )
    model.fit(train_df)
    return model

model = train_model(train_df, FEATURE_COLS, TARGET)
print("Model trained.")

Package 'snowflake-telemetry-python' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.
c:\Users\ArshiyaShaik\Downloads\MasterClass\LPDG_RGMCET_MASTERCLASS\.venv\Lib\site-packages\snowflake\ml\model\model_signature.py:74: UserWarning: The sample input has 1168 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


Model trained.


In [ ]:
test_df.write.mode("overwrite").save_as_table(
    "TBL_WEATHER_TEST_DATA"
)

### Save Test Data

Convert the Snowpark test DataFrame to a Pandas DataFrame and save it as a CSV file for later evaluation or analysis.

In [23]:
test_data_local = test_df.to_pandas()
test_data_local.to_csv("test_data.csv", index=False)
print(f"Saved {len(test_data_local)} rows to test_data.csv")

Saved 293 rows to test_data.csv


### Save Trained Model

Save the trained temperature forecasting model using Joblib so that it can be loaded and reused later without training the model again.

In [24]:
import joblib

joblib.dump(model, "temperature_model.joblib")
print("Model saved.")

Model saved.


## Summary

| Step | What it does | Function |
|---|---|---|
| 1. Load | Reads the raw table from Snowflake | `load_data()` |
| 2. Clean | 5 simple steps: fix types, handle missing values, remove duplicates, drop irrelevant columns, sort by time | `fix_data_types()`, `check_missing_values()`, `drop_missing_target()`, `check_and_remove_duplicates()`, `drop_irrelevant_columns()`, `sort_by_time()` |
| 3. Features + plots | Lag features for temperature *and* other sensors, hour of day, then visualizes the data | `add_features()`, `drop_warmup_rows()` |
| 4. Split | Chronological 80/20 train/test split | `train_test_split_by_time()` |
| 5. Train | Fits one Random Forest model on 9 features | `train_model()` |
| 6. Save the model |  saves the model using joblib | `joblib.dump()` |

**Why lag-1 of other sensors, not the raw columns:** this is still a *forecasting* setup, so
only information available before the hour being predicted is allowed in. `HUMIDITY_LAG_1`
is last hour's humidity — genuinely known in advance. Raw `HUMIDITY` (this hour's reading)
would be leakage, since you wouldn't have it yet in a real forecast.

**About the plots:** every plot pulls a small amount of data down from Snowpark with
`.to_pandas()` purely so Plotly can draw it. This is a visualization-only detour — the model
itself is trained directly on the Snowpark DataFrame, never on the pandas copy. When this
notebook becomes a stored procedure, the plotting cells (Step 3.1 and the bonus plot) are the
ones to leave out; the numbered pipeline functions carry the entire model.

**Note on structure:** every pipeline step above is a plain function that takes a Snowpark
DataFrame (or session) in and returns one out — no notebook-only state, no global side
effects beyond `session`. That's intentional: to turn this into a stored procedure later,
each function's body can go almost unchanged into a `CREATE OR REPLACE PROCEDURE ...
LANGUAGE PYTHON` block, with the session passed in as the procedure's handler argument.